### Embedding

- #### Logical routing
    A routing mechanism where the LLM chooses the DB based on the question asked.

- #### Semantic routing
    A routing mechanism where the question is embeded, and a prompt is chosen based on the similarity search.

In [ ]:
! pip -q install langchain_community tiktoken langchain-deepseek langchainhub chromadb langchain dotenv bs4 langchain-text-splitters langchain-ollama

In [ ]:
# Setup

import os
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings
from langchain_deepseek import ChatDeepSeek

load_dotenv()

# Loading my LLM API Key

EMBEDDING_MODEL_NAME = "qwen3-embedding:0.6b"
DEEPSEEK_MODEL_NAME='deepseek-chat'

OLLAMA_EMBEDDING = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)
DEEPSEEK_LLM = ChatDeepSeek(model=DEEPSEEK_MODEL_NAME, temperature=0, api_key=os.getenv('DEEPSEEK_API_KEY'))


## Overview:

In [ ]:
# Sample RAG Implemention
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma

urls = [
    'https://medlineplus.gov/ency/article/002458.htm',
    'https://pubmed.ncbi.nlm.nih.gov/40507137/',
    'https://pubmed.ncbi.nlm.nih.gov/41632571/'
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Removing unnecessary spaces and new lines
for doc in docs_list:
    doc.page_content = doc.page_content.replace('\n','').replace('  ', ' ').replace('\r', ' ')

# Embeddings
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

# Splitting
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=512, chunk_overlap=0)
docs_splits = text_splitter.split_documents(docs_list)

# Creating vector store
vector_store = Chroma.from_documents(
    documents = docs_splits,
    embedding = embeddings,
)

# Retriever used to retrieve from the vector database.
retriever = vector_store.as_retriever()

retrieved_docs = retriever.invoke("What are proteins?")

print(len(retrieved_docs))


## Deep Dive

In [ ]:

# Embedding a simple sentence

prompt = "On Friday I ate an apple watch while wearing an apple watch"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)
prompt_embeddings = embeddings.embed_documents(prompt)

# Resulting embedding:
print(f'- Dimension: {len(prompt_embeddings[0])} \n- Embedding:{prompt_embeddings[0]}', )

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma

# Loading document:
urls = [
    'https://medlineplus.gov/ency/article/002458.htm',
    'https://pubmed.ncbi.nlm.nih.gov/40507137/',
    'https://pubmed.ncbi.nlm.nih.gov/41632571/'
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Formating loaded document
for doc in docs_list:
    doc.page_content = doc.page_content.replace('\n','').replace('  ', ' ').replace('\r', ' ')


In [ ]:
# Embeddings
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

# Splitting, NB: The chunk size refers to token lenght of each chunk
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=512, chunk_overlap=0)
docs_splits = text_splitter.split_documents(docs_list)
len(docs_splits)

In [ ]:
# For visualization
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

for chunk in docs_splits:
    token_count = len(enc.encode(chunk.page_content))
    # chunk_embedding_vector = embeddings.embed_documents(chunk.page_content)
    print('Token count --->',  token_count)
    

In [ ]:
# Creating vector store
vector_store = Chroma.from_documents(
    documents = docs_splits,
    embedding = embeddings,
)

# Retriever used to retrieve from the vector database.
retriever = vector_store.as_retriever()

retrieved_docs = retriever.invoke("What are proteins?")

retrieved_docs

## Calculating cosine search.

cosine_similarity(A, B) = (A . B) / (||A|| * ||B||)

cosine_distance(A, B) = 1 - cosine_similarity(A, B)


In [ ]:
euclidean_search = vector_store.similarity_search_with_score("What are proteins?") # Euclidean search
for e_doc in euclidean_search:
    print(e_doc)

In [ ]:
cosine_search = vector_store.similarity_search_with_relevance_scores("What are proteins?") # Cosine search
for c_doc in cosine_search:
    print(c_doc)